<h1>TODO:</h1>
Handle food items with carbs and fiber count that do not make sense

In [1]:
# Import pandas library
import pandas as pd
import numpy as np
import dask.dataframe as dd

In [2]:
#   Define carb to fiber ratio category based on given value
def categorize_ratio(ratio):
    if ratio <= 3:
        return 'Best'
    elif 3 < ratio <= 5:
        return 'Ideal'
    elif 5 < ratio < 10:
        return 'Moderate'
    else:
        return 'Avoid'

In [3]:
pd.options.display.float_format = '{:.2f}'.format

<h1>NUTRITIONAL INFORMATION</h1>

In [4]:
#   Load the data with nutritional information
food = pd.read_csv("food_nutrient.csv")

In [5]:
#   We are interested in amount and nutrient id, so we remove all unwanted columns
columns_to_remove = ['data_points', 'derivation_id', 'min', 'max', 'median', 'footnote', 'min_year_acquired']
food = food.drop(columns=columns_to_remove)

In [6]:
# Print out food nutritional information
food.head()

,id,fdc_id,nutrient_id,amount
0,13706927,1105904,1257,0.00
1,13706930,1105904,1293,53.33
2,13706926,1105904,1253,0.00
3,13706921,1105904,1092,0.00
4,13706916,1105904,1008,867.00


In [7]:
print(food.dtypes)

id               int64
fdc_id           int64
nutrient_id      int64
amount         float64
dtype: object


In [12]:
#   Filter to check specific data with nonsensical values after preprocessing
df_filtered = food[food["fdc_id"] == 1814712]  # This creates a new filtered Dask DataFrame
#df_filtered_computed = df_filtered.compute()  # Compute to get a Pandas DataFrame

In [14]:
df_filtered.head(150)

,id,fdc_id,nutrient_id,amount
14933333,21403524,1814712,1003,0.00
14933334,21403529,1814712,1079,0.00
14933335,21403528,1814712,2000,3000.00
14933336,21403531,1814712,1253,0.00
14933337,21403527,1814712,1008,12000.00
14933338,21403530,1814712,1093,3000.00
14933339,21403526,1814712,1005,3000.00
14933340,21403525,1814712,1004,0.00


In [95]:
food.describe()

,id,fdc_id,nutrient_id,amount
count,25301180.00,25301180.00,25301180.00,25301180.00
mean,19311659.47,1599888.73,1171.71,802.98
std,8883172.24,687264.24,245.62,1518172.45
min,2684218.00,344604.00,1003.00,0.00
25%,13480101.75,1116882.00,1008.00,0.00
50%,19987620.50,1669860.50,1092.00,4.17
75%,26539421.25,2161720.00,1253.00,43.33
max,33849792.00,2688921.00,2068.00,7500000000.00


In [138]:
food.shape

(25301180, 4)

<h1>BRANDS INFORMATION</h1>

In [98]:
#   Load the data with brands information
brands = pd.read_csv("branded_food.csv", low_memory=False)

In [99]:
print("Brands shape:", brands.shape)

#   Brand name and owner as well as market country are the most important columns, we might also look at ingredients,
#   brand category and preparation state code
brand_columns_to_remove = ['subbrand_name', 'gtin_upc', 'not_a_significant_source_of', 'data_source', 'package_weight', 'modified_date', 'available_date', 'trade_channel', 'short_description', 'discontinued_date', 'household_serving_fulltext']
brands_cleaned = brands.drop(columns=brand_columns_to_remove)

Brands shape: (1958978, 20)


In [100]:
#   For this analysis, we want to know the brand or at least the brand owner
#   so we are getting rid of any rows that have missing data in both columns
missing_both_brandinfo = brands_cleaned[brands_cleaned['brand_owner'].isna() & brands_cleaned['brand_name'].isna()].shape[0]
print(missing_both_brandinfo)

# Remove rows where both 'brand_owner' and 'brand_name' are missing
brands_cleaned = brands_cleaned.dropna(subset=['brand_owner', 'brand_name'], how='all')

print("Brands shape after removing missing names and owner information:", brands_cleaned.shape)


1868
Brands shape after removing missing names and owner information: (1957110, 9)


In [139]:
brands_cleaned.tail()

,fdc_id,brand_owner,brand_name,ingredients,serving_size,serving_size_unit,branded_food_category,market_country,preparation_state_code
1958973,2688917,"Incobrasa Industries, Ltd.",Long Life,soybean oil,14.00,GRM,Oils Edible,United States,PREPARED
1958974,2688918,Tyson Foods Inc.,Advance Pierre Cab,"Beef, salt. Breaded and Battered with: enriche...",224.00,GRM,Meat/Poultry/Other Animals Prepared/Processed,United States,UNPREPARED
1958975,2688919,Tyson Foods Inc.,Advance Pierre Cab,"Beef, salt. Breaded and Battered with: enriche...",112.00,GRM,Meat/Poultry/Other Animals Prepared/Processed,United States,UNPREPARED
1958976,2688920,Tyson Foods Inc.,Steak-Eze,Beef. Containing up to 25% of a Solution of Wa...,112.00,GRM,Meat/Poultry/Other Animals Prepared/Processed,United States,UNPREPARED
1958977,2688921,Clemens Food Group LLC,Hatfield,Pig Head,100.00,GRM,Meat/Poultry/Other Animals Unprepared/Unproce...,United States,UNPREPARED


In [140]:
# Get the number of unique values in a specific column
unique_brand_cat_count = brands_cleaned['branded_food_category'].nunique()
print(f"Seems like we have '{unique_brand_cat_count}' unique brand categories.")

Seems like we have '428' unique brand categories.


<h1>NUTRIENT INFORMATION</h1>

Main purpose is to calculate Carb to Fiber ratios and categorize based on results.  It was intended originally to also look
at Sugar amounts, but this was discarded due to lack of data with sugar information.

In [103]:
#   Load the data that contains all the nutrients considered in the nutritional information
nutrients = pd.read_csv("nutrient.csv")
nutrients.shape

(477, 5)

In [104]:
nutrients.head(10)

,id,name,unit_name,nutrient_nbr,rank
0,2047,Energy (Atwater General Factors),KCAL,957.00,280.00
1,2048,Energy (Atwater Specific Factors),KCAL,958.00,290.00
2,1001,Solids,G,201.00,200.00
3,1002,Nitrogen,G,202.00,500.00
4,1003,Protein,G,203.00,600.00
5,1004,Total lipid (fat),G,204.00,800.00
6,1005,"Carbohydrate, by difference",G,205.00,1110.00
7,1006,"Fiber, crude (DO NOT USE - Archived)",G,206.00,999999.00
8,1007,Ash,G,207.00,1000.00
9,1008,Energy,KCAL,208.00,300.00


In [111]:
nutrients['name']

0                Energy (Atwater General Factors)
1               Energy (Atwater Specific Factors)
2                                          Solids
3                                        Nitrogen
4                                         Protein
                          ...                    
472                              Oligosaccharides
473    Low Molecular Weight Dietary Fiber (LMWDF)
474                                     Vitamin E
475                                     Vitamin A
476                                   Glutathione
Name: name, Length: 477, dtype: object

In [112]:
# Filter rows where 'name' contains 'Carbohydrate' or 'Carbohydrates' (case-insensitive)
# Do the same for Fiber
carb_df = nutrients[nutrients['name'].str.contains("Carbohydrate|Carbohydrates", case=False, na=False)]

# Display the filtered rows
print(carb_df)

       id                         name unit_name  nutrient_nbr    rank
6    1005  Carbohydrate, by difference         G        205.00 1110.00
51   1050   Carbohydrate, by summation         G        205.20 1120.00
73   1072          Carbohydrate, other         G        284.00     NaN
450  2039                Carbohydrates         G        956.00 1100.00


Follow the nutrient ids:

Carbohydrates: 2039 Carbohydrates, 1005 Carbohydrate by difference, 1072 Other
Protein: 1003 Protein
Fat: 1004 Total lipid (fat), 1085 Total fat (NLEA)
Fiber: 1079 Fiber - total dietary, 2033 Total dietary fiber (AOAC 2011.25)
Sugars: 1063 Sugars - Total, 2000 Total Sugars
Added Sugars: 1235 Sugars, added


In [113]:
# List of relevant nutrient IDs
keep_ids = [1003, 1004, 1005, 1053, 1063, 1079, 1085, 1235, 2000, 2033, 2039, 1072]

# Filter the dataset
filtered_df = nutrients[nutrients['id'].isin(keep_ids)]

print(filtered_df)  # Display result


       id                                name unit_name  nutrient_nbr    rank
4    1003                             Protein         G        203.00  600.00
5    1004                   Total lipid (fat)         G        204.00  800.00
6    1005         Carbohydrate, by difference         G        205.00 1110.00
54   1053                    Adjusted Protein         G        257.00  700.00
64   1063                       Sugars, Total         G        269.30 1500.00
73   1072                 Carbohydrate, other         G        284.00     NaN
80   1079                Fiber, total dietary         G        291.00 1200.00
86   1085                    Total fat (NLEA)         G        298.00  900.00
236  1235                       Sugars, added         G        539.00 1540.00
415  2000                        Total Sugars         G        269.00 1510.00
444  2033  Total dietary fiber (AOAC 2011.25)         G        293.00 1300.00
450  2039                       Carbohydrates         G        9

In [141]:
filtered_df.shape

(12, 5)

In [142]:
# Filter the nutritional information dataset to contain only the needed nutrients
filtered_food_df = food[food['nutrient_id'].isin(keep_ids)]

print(filtered_food_df.shape)  # Display result

(9311545, 4)


<h1>FOOD NUTRITION AND BRANDS PROCESSING</h1>

The ratios will be categorized based on the following:
<ul>
<li><strong>Best:</strong>   3:1 or lower</li>
<li><strong>Ideal:</strong>  5:1 or lower</li>
<li><strong>Moderate:</strong>   9:1 or lower</li>
<li><strong>Avoid:</strong>   10:1 or greater</li>
</ul>

In [116]:
# Filter rows where 'id' contains 'Carbohydrate by difference' id = 1005
food_df = food[food['nutrient_id'].isin([1005, 1079])]

# Display the filtered rows
#print(food_df)
is_column_A_zero = (food_df['amount'] == 0).all()
zero_count = (food_df['amount'] == 0).sum()
print('Shape ', food_df.shape)
print('Is column zero? ', is_column_A_zero)
print('Zero Count ', zero_count)

Shape  (3382838, 4)
Is column zero?  False
Zero Count  730316


In [117]:
#   Grab only the data that has amounts different than 0
#   Because of the size of the data set, we want to work only with
#   foods that have both carbs and fiber.
df_food_filtered = food_df[food_df['amount'] != 0]
df_food_filtered.shape

(2652522, 4)

In [143]:
df_food_filtered.head()

,id,fdc_id,nutrient_id,amount
21,13706207,1105905,1005,0.42
33,13706300,1105906,1005,6.12
36,13706303,1105906,1079,0.40
50,13705763,1105907,1079,0.40
55,13705760,1105907,1005,5.31


In [144]:
#   For future processing, we need this dataframe to be ordered in a way that for each
#   brand, the carb information is always first, and the fiber is always second
df_food_filtered_sorted = df_food_filtered.sort_values(by=['fdc_id', 'nutrient_id'])

#   While working on this, we found out that some nutrient amounts are repeated
#   and some have different values twice.  To handle this, we are removing the repeated
#   amounts, still deciding what to do with the ones that have different values
#   (need to find out which is the correct information or remove)
#   Remove rows with duplicate 'fdc_id', 'nutrient_id', and 'amount'
df_food_filtered_sorted = df_food_filtered_sorted.drop_duplicates(subset=['fdc_id', 'nutrient_id', 'amount'])

# Check shape after removing duplicates
print(df_food_filtered_sorted.shape)


(2652481, 4)


In [145]:
#   Lets try joining the data frames before grouping the nutrient values
#   Use Dask library to improve performance given that current data has millions of rows
dd_brands = dd.from_pandas(brands_cleaned, npartitions=10)
dd_nutrition = dd.from_pandas(df_food_filtered_sorted, npartitions=10)

# Perform an inner join (only matching rows are kept, to remove brands without any nutrient information)
result = dd_brands.merge(dd_nutrition, on='fdc_id', how='inner')

# Compute the result
df_merged_data = result.compute()

In [146]:
df_merged_data.head(10)

,fdc_id,brand_owner,brand_name,ingredients,serving_size,serving_size_unit,branded_food_category,market_country,preparation_state_code,id,nutrient_id,amount
0,2248918,Stater Bros. Markets Inc.,STATER BROS.,"POPCORN, PALM OIL, CONTAINS 2% OR LESS OF EACH...",33.00,g,"Popcorn, Peanuts, Seeds & Related Snacks",United States,<NA>,27672180,1079,9.10
1,2248918,Stater Bros. Markets Inc.,STATER BROS.,"POPCORN, PALM OIL, CONTAINS 2% OR LESS OF EACH...",33.00,g,"Popcorn, Peanuts, Seeds & Related Snacks",United States,<NA>,27672177,1005,51.52
2,2248982,"Wegmans Food Markets, Inc.",WEGMANS,"BROCCOLI FLORETS, SUGAR SNAP PEAS, CARROTS, BO...",85.00,g,Frozen Vegetables,United States,<NA>,27664203,1079,2.40
3,2248982,"Wegmans Food Markets, Inc.",WEGMANS,"BROCCOLI FLORETS, SUGAR SNAP PEAS, CARROTS, BO...",85.00,g,Frozen Vegetables,United States,<NA>,27664200,1005,5.88
4,2248998,"Tops Markets, LLC",TOPS,"POTATOES, WATER, SALT, CALCIUM CHLORIDE.",165.00,g,Canned Vegetables,United States,<NA>,27663458,1079,1.20
5,2248998,"Tops Markets, LLC",TOPS,"POTATOES, WATER, SALT, CALCIUM CHLORIDE.",165.00,g,Canned Vegetables,United States,<NA>,27663455,1005,8.48
6,2249001,"Tops Markets, LLC",TOPS,"PEAS, WATER, SUGAR, SALT.",125.00,g,Canned Vegetables,United States,<NA>,27663473,1079,2.40
7,2249001,"Tops Markets, LLC",TOPS,"PEAS, WATER, SUGAR, SALT.",125.00,g,Canned Vegetables,United States,<NA>,27663470,1005,9.60
8,2249062,"Tops Markets, LLC",TOPS,"TOMATO CONCENTRATE, HIGH FRUCTOSE CORN SYRUP, ...",17.00,g,"Ketchup, Mustard, BBQ & Cheese Sauce",United States,<NA>,27663702,1005,29.41
9,2249148,"Kahiki Foods, Inc.",KAHIKI,"BOWL [FRIED RICE (COOKED RICE, CARROTS, RED BE...",340.00,g,Frozen Dinners & Entrees,United States,<NA>,27706381,1005,21.18


In [147]:
df_merged_data.shape

(2650831, 12)

In [148]:
# Pivot the data to separate value columns for nutrient_id = 1005 (Carbs) and nutrient_id = 1079 (Fiber)
df_pivot = df_merged_data.pivot_table(index='fdc_id', columns='nutrient_id', values='amount', aggfunc='first')

# Rename columns for better clarity (df2_id = 1 -> 'value1', df2_id = 2 -> 'value2')
df_pivot = df_pivot.rename(columns={1005: 'Carbs', 1079: 'Fiber'})

# Fill missing values with 0 where a df2_id does not exist for a given df1_id
df_pivot = df_pivot.fillna(0)

df_pivot.head(10)

nutrient_id,Carbs,Fiber
fdc_id,,
344604,4.07,0.80
344605,4.07,0.80
344609,72.70,1.30
344610,73.10,1.20
344611,73.00,1.30
344612,57.00,1.90
344613,57.00,1.90
344614,60.10,2.10
344615,60.10,2.10


In [149]:
#   Calculate the ratio, assigning a negligible amount of Fiber by default when this is 0
#   to reflect a lack of fiber (assuming/highlighting foods without fiber even if it is data error)
#   df_pivot["Ratio"] = df_pivot["Carbs"] / df_pivot["Fiber"].replace(0, np.nan)
df_pivot["Ratio"] = np.where(
    (df_pivot["Fiber"] == 0) | df_pivot["Fiber"].isna(), 
    df_pivot["Carbs"], 
    df_pivot["Carbs"] / df_pivot["Fiber"]
)

#   Categorize the ratio based on the value obtained
df_pivot["Category"] = df_pivot["Ratio"].apply(categorize_ratio)

In [150]:
df_pivot

nutrient_id,Carbs,Fiber,Ratio,Category
fdc_id,,,,
344604,4.07,0.80,5.09,Moderate
344605,4.07,0.80,5.09,Moderate
344609,72.70,1.30,55.92,Avoid
344610,73.10,1.20,60.92,Avoid
344611,73.00,1.30,56.15,Avoid
...,...,...,...,...
2688915,25.06,0.00,25.06,Avoid
2688916,31.00,16.00,1.94,Best
2688918,24.81,1.10,22.55,Avoid


In [151]:
avoid_count = df_pivot[df_pivot['Category'] == 'Avoid'].shape[0]
best_count = df_pivot[df_pivot['Category'] == 'Best'].shape[0]
ideal_count = df_pivot[df_pivot['Category'] == 'Ideal'].shape[0]
moderate_count = df_pivot[df_pivot['Category'] == 'Moderate'].shape[0]

print("Best Count: ", best_count)
print("Ideal Count: ", ideal_count)
print("Moderate Count: ", moderate_count)
print("Avoid Count: ", avoid_count)


Best Count:  192104
Ideal Count:  208851
Moderate Count:  314911
Avoid Count:  944906


In [152]:
#   Now create a new dataframe by  merging original brands data with the 
#   new data that has Carbs, Fiber, Ratio and Categorized Ratio information
#   Lets try joining the data frames before grouping the nutrient values
dd_pivot = dd.from_pandas(df_pivot, npartitions=10)

# Perform an inner join (only matching rows are kept)
final_result = dd_brands.merge(dd_pivot, on='fdc_id', how='inner')

# Compute the result (Dask is lazy, so you need to call compute() to get the result)
df_final_merged_data = final_result.compute()

In [153]:
df_final_merged_data.head(10)

,fdc_id,brand_owner,brand_name,ingredients,serving_size,serving_size_unit,branded_food_category,market_country,preparation_state_code,Carbs,Fiber,Ratio,Category
0,1792518,Family Dollar Stores Inc.,MIDWOOD BRANDS,"SUGAR, GLUCOSE SYRUP, WATER, CITRIC ACID, ARTI...",16.00,g,Candy,United States,<NA>,93.75,0.00,93.75,Avoid
1,1792531,"Twang Partners, Ltd.",TWANG,"SEA SALT, CANE SUGAR, CITRIC ACID, SPICES, CHI...",1.20,g,"Seasoning Mixes, Salts, Marinades & Tenderizers",United States,<NA>,83.33,0.00,83.33,Avoid
2,1792556,"Taylor Fresh Foods, Inc.",<NA>,"BEEF BOLOGNA (BEEF, WATER, CORN SYRUP, DEXTROS...",136.00,g,Prepared Subs & Sandwiches,United States,<NA>,20.59,0.70,29.41,Avoid
3,1792571,Bi-Lo Inc.,SE GROCERS,"SKIM MILK, CREAM, SUGAR, CORN SYRUP, COCOA PRO...",68.00,g,Ice Cream & Frozen Yogurt,United States,<NA>,25.00,1.50,16.67,Avoid
4,1792591,"Bay Valley Foods, LLC",GOODFIELDS,WALNUTS.,30.00,g,"Popcorn, Peanuts, Seeds & Related Snacks",United States,<NA>,13.33,6.70,1.99,Best
5,1792604,"Bay Valley Foods, LLC",GOODFIELDS,"GLUTINOUS RICE, SOY SAUCE (WATER, SOYBEANS, WH...",30.00,g,Other Snacks,United States,<NA>,83.33,6.70,12.44,Avoid
6,1792637,"Associated Food Stores, Inc.",RED BUTTON VINTAGE CREAMERY,"MILK, CANE SUGAR, CREAM, COCONUT OIL, COCOA BU...",93.00,g,Ice Cream & Frozen Yogurt,United States,<NA>,24.73,1.10,22.48,Avoid
7,1792647,"Frankford Candy, LLC",<NA>,"SUGAR, GLUCOSE SYRUP, CITRIC ACID, ARTIFICIAL ...",14.00,g,Candy,United States,<NA>,100.00,0.00,100.00,Avoid
8,1792672,"Bay Valley Foods, LLC",GOODFIELDS,"SUNFLOWER KERNELS, PEANUT OIL AND/OR SUNFLOWER...",30.00,g,"Popcorn, Peanuts, Seeds & Related Snacks",United States,<NA>,23.33,10.00,2.33,Best
9,1792685,"Bay Valley Foods, LLC",GOODFIELDS,"PEANUTS (PEANUTS, PEANUT OIL, SALT), PEANUT BU...",28.00,g,"Popcorn, Peanuts, Seeds & Related Snacks",United States,<NA>,50.00,7.10,7.04,Moderate


In [155]:
print(df_final_merged_data.dtypes)

fdc_id                              int64
brand_owner               string[pyarrow]
brand_name                string[pyarrow]
ingredients               string[pyarrow]
serving_size                      float64
serving_size_unit         string[pyarrow]
branded_food_category     string[pyarrow]
market_country            string[pyarrow]
preparation_state_code    string[pyarrow]
Carbs                             float64
Fiber                             float64
Ratio                             float64
Category                  string[pyarrow]
dtype: object


In [156]:
#   Check for number of unique values in some category columns for
#   future processing
# Assuming your Dask DataFrame is loaded into 'df'
unique_values = df_final_merged_data['preparation_state_code'].nunique()
u_values = df_final_merged_data['preparation_state_code'].unique()

# To compute the result (since Dask is lazy):
print(unique_values)  # This triggers the computation
print(u_values)  # This triggers the computation

17
<ArrowStringArray>
[            <NA>,       'PREPARED',   'READY_TO_EAT',           'THAW',
     'UNPREPARED',     'CONVECTION', 'READY_TO_DRINK', 'HEAT_AND_SERVE',
         'FREEZE',           'BAKE',            'FRY',          'STEAM',
            'DRY',           'BOIL',      'MICROWAVE',    'UNSPECIFIED',
       'DEEP_FRY',       'STIR_FRY']
Length: 18, dtype: string


In [157]:
#   Check for number of unique values in some category columns for
#   future processing
# Assuming your Dask DataFrame is loaded into 'df'
unique_values1 = df_final_merged_data['market_country'].nunique()
u_values1 = df_final_merged_data['market_country'].unique()

# To compute the result (since Dask is lazy):
print(unique_values1)  # This triggers the computation
print(u_values1)  # This triggers the computation

2
<ArrowStringArray>
['United States', 'New Zealand']
Length: 2, dtype: string


In [ ]:
#   Check for number of unique values in some category columns for
#   future processing
# Assuming your Dask DataFrame is loaded into 'df'
unique_values2 = df_final_merged_data['branded_food_category'].nunique()
u_values2 = df_final_merged_data['branded_food_category'].unique()

# To compute the result (since Dask is lazy):
print(unique_values2)  # This triggers the computation
print(u_values2)  # This triggers the computation

In [159]:
df_final_merged_data.shape

(1660772, 13)

In [ ]:
#   Query the data frame to look at food from New Zealand
df_filtered = df_final_merged_data.query('market_country == "New Zealand"')

df_filtered

,fdc_id,brand_owner,brand_name,ingredients,serving_size,serving_size_unit,branded_food_category,market_country,Carbs,Fiber,Ratio,Category
38772,2316821,BERKANO ORGANICS LIMITED,Berkano Foods,"Tomato Paste, Tortilla (Wheat Flour, Water, Ve...",400.00,g,Prepared Meals,New Zealand,16.00,0.00,16.00,Avoid
38773,2316828,SPEIRS FOODS (2018) LP,Speirs Foods,"Potato, Dressing (Water, Canola Oil , Sour Cre...",100.00,g,Salads,New Zealand,11.70,0.00,11.70,Avoid
38774,2316830,ITALIAN CHEESES LIMITED,Massimo's Italian Cheeses,"Pasteurised Cow's Milk, Citric Acid, Salt, Veg...",62.50,g,Cheese - Speciality,New Zealand,0.30,0.00,0.30,Best
38775,2316833,ITALIAN CHEESES LIMITED,Massimo's,"Pasteurised Cow's Milk, Acidity Regulator (Cit...",62.50,g,Cheese - Speciality,New Zealand,0.30,0.00,0.30,Best
38776,2316840,TEGEL FOODS LIMITED,Tegel,"Chicken, Water, Maltodextrin, Salt, Mineral Sa...",150.00,g,Frozen Chicken - Processed,New Zealand,0.67,0.00,0.67,Best
...,...,...,...,...,...,...,...,...,...,...,...,...
77438,2183102,No Brand Owner supplied for New Zealand Data,Kellogg's,"Rice Bubbles (Rice, Sugar, Salt, Barley Malt E...",22.00,g,Wrapped Snacks - Cereal,New Zealand,75.00,1.40,53.57,Avoid
77439,2183113,No Brand Owner supplied for New Zealand Data,Market Value,"Pork, Water, Seasoning (Rice Flour, Potato Sta...",64.40,g,Sausages/Smallgoods,New Zealand,5.75,0.00,5.75,Moderate
77440,2183116,No Brand Owner supplied for New Zealand Data,Tasti,"Nuts (Peanuts, Roasted Almonds, Cashews), Gluc...",35.00,g,Wrapped Snacks - Nut Bars,New Zealand,36.57,0.00,36.57,Avoid
77441,2183134,No Brand Owner supplied for New Zealand Data,Bakels,"Sugar, Wheat Flour, Vegetable Oil, Egg Powder,...",50.00,g,Baking Needs,New Zealand,69.60,2.00,34.80,Avoid


In [171]:
#   We are only looking at food from United States, and fortunately the New Zealand
#   data is only 1033 rows, so we are removing those
df_final_merged_data = df_final_merged_data[df_final_merged_data['market_country'] != 'New Zealand']
df_final_merged_data.shape

(1659739, 12)

In [172]:
# Check for missing values in a specific column (e.g., 'preparation_state_code')
missing_values_count = df_final_merged_data.isnull().sum()

# Compute and print the number of missing values
missing_values_count

fdc_id                        0
brand_owner               13914
brand_name               478439
ingredients                2815
serving_size                  0
serving_size_unit          7437
branded_food_category      8443
market_country                0
Carbs                         0
Fiber                         0
Ratio                         0
Category                      0
dtype: int64

In [167]:
#   Unfortunately, preparation state code has the mayority of missing
#   data, so this column will not be useful for this study
df_final_merged_data = df_final_merged_data.drop(['preparation_state_code'], axis=1)

In [173]:
df_final_merged_data

,fdc_id,brand_owner,brand_name,ingredients,serving_size,serving_size_unit,branded_food_category,market_country,Carbs,Fiber,Ratio,Category
0,1792518,Family Dollar Stores Inc.,MIDWOOD BRANDS,"SUGAR, GLUCOSE SYRUP, WATER, CITRIC ACID, ARTI...",16.00,g,Candy,United States,93.75,0.00,93.75,Avoid
1,1792531,"Twang Partners, Ltd.",TWANG,"SEA SALT, CANE SUGAR, CITRIC ACID, SPICES, CHI...",1.20,g,"Seasoning Mixes, Salts, Marinades & Tenderizers",United States,83.33,0.00,83.33,Avoid
2,1792556,"Taylor Fresh Foods, Inc.",<NA>,"BEEF BOLOGNA (BEEF, WATER, CORN SYRUP, DEXTROS...",136.00,g,Prepared Subs & Sandwiches,United States,20.59,0.70,29.41,Avoid
3,1792571,Bi-Lo Inc.,SE GROCERS,"SKIM MILK, CREAM, SUGAR, CORN SYRUP, COCOA PRO...",68.00,g,Ice Cream & Frozen Yogurt,United States,25.00,1.50,16.67,Avoid
4,1792591,"Bay Valley Foods, LLC",GOODFIELDS,WALNUTS.,30.00,g,"Popcorn, Peanuts, Seeds & Related Snacks",United States,13.33,6.70,1.99,Best
...,...,...,...,...,...,...,...,...,...,...,...,...
166234,832174,Food Town Stores Inc.,<NA>,"CARBONATED WATER, KIWI JUICE CONCENTRATE, NATU...",503.00,ml,Water,United States,0.40,0.00,0.40,Best
166235,832180,Food Town Stores Inc.,<NA>,"CARBONATED WATER, HIGH FRUCTOSE CORN SYRUP, CA...",360.00,ml,Soda,United States,11.67,0.00,11.67,Avoid
166236,832202,"GIANT Snacks, Inc.",<NA>,"NON-GMO SUNFLOWER SEEDS, SEA SALT, SUGAR, SEAS...",30.00,g,"Popcorn, Peanuts, Seeds & Related Snacks",United States,20.00,13.30,1.50,Best
166237,832210,Ferris Coffee & Nut Co.,<NA>,"ORIENTAL MIX (GLUTINOUS RICE, SOY SAUCE [WATER...",30.00,g,Other Snacks,United States,50.00,3.30,15.15,Avoid


In [164]:
# Filter to view rows where 'preparation_state_code' is missing
missing_data_df = df_final_merged_data[df_final_merged_data['branded_food_category'].isnull()]

# Compute and print the rows with missing values in that column
missing_data_df


,fdc_id,brand_owner,brand_name,ingredients,serving_size,serving_size_unit,branded_food_category,market_country,preparation_state_code,Carbs,Fiber,Ratio,Category
20731,344605,Red Gold,<NA>,"Tomatoes, Tomato Juice, Less Than 2% Of: Salt,...",123.00,g,<NA>,United States,<NA>,4.07,0.80,5.09,Moderate
20827,345537,Red Gold,<NA>,"Tomato Puree (Water, Tomato Paste), Water, Les...",61.00,g,<NA>,United States,<NA>,6.56,1.60,4.10,Ideal
20828,345540,Red Gold,<NA>,"Tomatoes, Tomato Juice, Sugar, Salt, Dried Oni...",123.00,g,<NA>,United States,<NA>,6.50,0.80,8.12,Moderate
20829,345557,Red Gold,<NA>,"Tomato Puree (Water, Tomato Paste), Water, Les...",61.00,g,<NA>,United States,<NA>,6.56,1.60,4.10,Ideal
20830,345560,Cargill,<NA>,"Turkey, water, contains less than 2% salt, dex...",70.00,g,<NA>,United States,<NA>,1.00,0.00,1.00,Best
...,...,...,...,...,...,...,...,...,...,...,...,...,...
163853,763246,"BEAVER STREET FISHERIES, INC.",<NA>,"Scallops, bleached wheat flour, enriched bleac...",113.00,g,<NA>,United States,<NA>,30.09,0.90,33.43,Avoid
163883,764158,"BEAVER STREET FISHERIES, INC.",<NA>,"Scallops, water.",113.00,g,<NA>,United States,<NA>,3.54,0.00,3.54,Ideal
163950,765798,Beaver Street Fisheries Inc.,<NA>,Squid,112.00,g,<NA>,United States,<NA>,2.68,0.00,2.68,Best
163992,767170,Beaver Street Fisheries Inc.,<NA>,"Scallops, water, sodium tripolyphosphate.",113.00,g,<NA>,United States,<NA>,3.54,0.00,3.54,Ideal


In [174]:
#   Save the merged data into a csv file for further processing
#   Create a new csv file with the processed brands 
df_final_merged_data.to_csv('categorized_ratios_output.csv', index=True)